# This script reports UDN subject with phenotype and a list of HPO terms.

In [ ]:
import requests
import pandas as pd
import os
from datetime import datetime
from time import sleep
import pytz
from fhir_fetcher import fetch_all_data  # Ensure this module is available and handles paging through all records


# Function to fetch patient IDs from ResearchSubject API with an optional count
def fetch_patient_ids(session, query_url, num_pages=0, verbose='n'):
    """
    Fetches patient IDs from the given ResearchSubject API query URL.
    Optionally limits the number of patient IDs returned.

    Parameters:
        session: requests.Session object with headers configured.
        query_url: str, API URL to fetch patient IDs.
        count: int, optional, maximum number of patient IDs to fetch.

    Returns:
        List of patient IDs (strings).
    """
    research_subjects = fetch_all_data(session, query_url, num_pages, verbose)
    patient_ids = [entry['resource']['individual']['reference'].split('/')[-1] for entry in research_subjects]
    return patient_ids

# Function to fetch Observations for a set of patient IDs
# Function to fetch Observations for a set of patient IDs
def fetch_observations(session, patient_ids):
    """
    Fetch observations for a set of patient IDs using the fetch_all_data function.
    """
    patient_id_str = ",".join([f"Patient/{pid}" for pid in patient_ids])
    obs_url = f"{fhir_base_url}/Observation?subject={patient_id_str}"
    
    print(f"Fetching observations from URL: {obs_url}")
    
    try:
        # Use fetch_all_data to handle pagination
        all_entries = fetch_all_data(session, obs_url)
        observations = []
        
        for entry in all_entries:
            resource = entry.get('resource', {})
            obs_code = resource.get('code', {}).get('coding', [{}])[0].get('code', '')
            obs_display = resource.get('code', {}).get('coding', [{}])[0].get('display', '')
            valueString = resource.get('valueString', '')
            patient_ref = resource.get('subject', {}).get('reference', '')
            patient_id = patient_ref.split('/')[-1]
            
            observations.append({
                'patient_id': patient_id,
                'Obs.Code': obs_code,
                'Obs.Display': obs_display,
                'Obs.Value': valueString
            })
        
        return observations
    except requests.exceptions.RequestException as e:
        print(f"Error fetching Observations: {e}")
        return []

######################################333
# Define the mapping for PRIMARY_SYMPTOM_CATEGORY values
primary_symptom_mapping = {
    "1": "Allergies and Disorders of The Immune System",
    "2": "Cardiology and vascular conditions (heart, artery, vein, and lymph disorders)",
    "3": "Dentistry and craniofacial (Bones of head and face)",
    "4": "Dermatology (Skin diseases and disorders)",
    "5": "Endocrinology (Disorder of the endocrine glands and hormones)",
    "6": "Gastroenterology (Disorder of the stomach and intestines)",
    "8": "Hematology (Blood diseases and disorders)",
    "10": "Musculoskeletal and orthopedics (Structural and functional disorders of muscles, bones, and joints)",
    "11": "Nephrology (Kidney diseases and disorders)",
    "12": "Neurology (Disorders of the nervous system, including brain and spinal cord)",
    "14": "Ophthalmology (Eye disorders and diseases)",
    "16": "Pulmonology (Lung disorders and diseases)",
    "17": "Rheumatology (Immune disorders of the joints, muscles, and ligaments)",
    "20": "N/A",
    "22": "Other"
}

# Function to map PRIMARY_SYMPTOM_CATEGORY value to its corresponding description
def map_primary_symptom_category(value):
    return primary_symptom_mapping.get(value, value)  # Return the description if it exists, otherwise the original value


######################################333

# Function to fetch Conditions and group HPO terms for each patient
# Function to fetch Conditions and group HPO terms for each patient
def fetch_conditions_grouped(session, patient_ids):
    """
    Fetch conditions and group HPO terms for each patient using the fetch_all_data function.
    """
    patient_id_str = ",".join([f"Patient/{pid}" for pid in patient_ids])
    cond_url = f"{fhir_base_url}/Condition?subject={patient_id_str}"
    
    print(f"Fetching conditions from URL: {cond_url}")
    
    try:
        # Use fetch_all_data to handle pagination
        all_entries = fetch_all_data(session, cond_url)
        conditions = {}
        
        for entry in all_entries:
            resource = entry.get('resource', {})
            hpo_code = resource.get('code', {}).get('coding', [{}])[0].get('code', '')
            patient_ref = resource.get('subject', {}).get('reference', '')
            patient_id = patient_ref.split('/')[-1]
            
            if patient_id not in conditions:
                conditions[patient_id] = []
            conditions[patient_id].append(hpo_code)
        
        # Convert HPO terms to a comma-separated string
        conditions_grouped = [
            {'patient_id': pid, 'HPO_Terms': ', '.join(terms)}
            for pid, terms in conditions.items()
        ]
        
        return conditions_grouped
    except requests.exceptions.RequestException as e:
        print(f"Error fetching Conditions: {e}")
        return []



# Function to pivot observations (without the "Obs_" prefix)
def pivot_observations(observations):
    df_obs = pd.DataFrame(observations)
    if not df_obs.empty:
        # Apply mapping for PRIMARY_SYMPTOM_CATEGORY where applicable
        df_obs['Obs.Value'] = df_obs.apply(
            lambda row: map_primary_symptom_category(row['Obs.Value']) 
            if row['Obs.Display'] == "PRIMARY_SYMPTOM_CATEGORY" 
            else row['Obs.Value'], 
            axis=1
        )
        
        pivot_obs = df_obs.pivot_table(
            index='patient_id',
            columns='Obs.Display',
            values='Obs.Value',
            aggfunc='first'
        )
        return pivot_obs.reset_index()
    else:
        return pd.DataFrame()



# Combine observations and conditions into one DataFrame
def combine_data_with_hpo(observations, conditions_grouped):
    df_obs = pd.DataFrame(observations)
    df_cond = pd.DataFrame(conditions_grouped)
    
    # Merge observations and conditions on patient_id
    if not df_obs.empty and not df_cond.empty:
        combined_data = pd.merge(df_obs, df_cond, on='patient_id', how='outer')
    elif not df_obs.empty:
        combined_data = df_obs
    elif not df_cond.empty:
        combined_data = df_cond
    else:
        combined_data = pd.DataFrame()
    
    return combined_data

# Save the combined data to CSV
def save_to_csv(dataframe, file_name):
    dataframe.to_csv(file_name, index=False)
    print(f"Data saved to {file_name}")

# Main execution
TST_PATH = 'task-specific-token-all.txt'
fhir_base_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot2/x1"
# fhir-jpa-pilot at https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1 is the test server with synthetic data, with no token required.
fhir_base_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1"

with open(os.path.expanduser(TST_PATH), 'r') as f:
    tst_token = f.read().strip()

session = requests.Session()
session.headers.update({
    'Accept': 'application/fhir+json',
    'Authorization': f'Bearer {tst_token}',
    'Content-Type': 'application/x-www-form-urlencoded',
})

patient_query_url = f"{fhir_base_url}/ResearchSubject?study=ResearchStudy/phs001232"

# Fetch patient IDs
# Updated main execution block

# Fetch patient IDs
local_timezone = pytz.timezone('America/New_York')
starttime = datetime.now(local_timezone)
print("====== Start Time:", starttime.strftime('%Y-%m-%d %H:%M:%S %Z%z'))

patient_ids = fetch_patient_ids(session, patient_query_url, 0, 'n')
print(f"Total patients fetched: {len(patient_ids)}")

# Fetch data in chunks
chunk_size = 200
chunks = [patient_ids[i:i + chunk_size] for i in range(0, len(patient_ids), chunk_size)]

all_observations = []
all_conditions_grouped = []

for chunk in chunks:
    observations = fetch_observations(session, chunk)
    sleep(0.3)
    conditions_grouped = fetch_conditions_grouped(session, chunk)
    sleep(0.3)
    all_observations.extend(observations)
    all_conditions_grouped.extend(conditions_grouped)

# Pivot observations
pivot_obs = pivot_observations(all_observations)

# Combine data
combined_data = combine_data_with_hpo(pivot_obs, all_conditions_grouped)

# 1. Save Observation Data (Without HPO Terms)
print("\n====== Observation Data (Without HPO Terms) ======\n")
if not pivot_obs.empty:
    print(pivot_obs.head(10))
    save_to_csv(pivot_obs, "observation_data.csv")
else:
    print("No observation data available.")

# 2. Save Patient IDs with HPO Terms
print("\n====== Patient IDs with HPO Terms ======\n")
if all_conditions_grouped:
    df_conditions = pd.DataFrame(all_conditions_grouped)
    print(df_conditions[['patient_id', 'HPO_Terms']].head(10))
    save_to_csv(df_conditions[['patient_id', 'HPO_Terms']], "patient_hpo_terms.csv")
else:
    print("No HPO terms available.")

# 3. Save Combined Result (Observations and HPO Terms)
print("\n====== Combined Result (Observations and HPO Terms) ======\n")
if not combined_data.empty:
    print(combined_data.head(10))
    save_to_csv(combined_data, "combined_data_with_hpo.csv")
else:
    print("No combined data available.")



====== Start Time: 2025-01-07 16:23:46 EST-0500
Total patients fetched: 5837
Fetching observations from URL: https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/Observation?subject=Patient/1827029,Patient/1782246,Patient/1827106,Patient/1782249,Patient/1826993,Patient/1827049,Patient/1827014,Patient/1827102,Patient/1826982,Patient/1827072,Patient/1827065,Patient/1827094,Patient/1827020,Patient/1826986,Patient/1782244,Patient/1827063,Patient/1826991,Patient/1826983,Patient/1827032,Patient/1827002,Patient/1827026,Patient/1826994,Patient/1827100,Patient/1827037,Patient/1827062,Patient/1827023,Patient/1826979,Patient/1827048,Patient/1827054,Patient/1827095,Patient/1827059,Patient/1827101,Patient/1827006,Patient/1827028,Patient/1782243,Patient/1827086,Patient/1827060,Patient/1827001,Patient/1827066,Patient/1827096,Patient/1827075,Patient/1826996,Patient/1782253,Patient/1827033,Patient/1827085,Patient/1827013,Patient/1826989,Patient/1827024,Patient/1827074,Patient/1827078,Patient/1782251,Pa